## 上下文管理器
with 语句、`__enter__` / `__exit__`、contextlib

### 1. 为什么需要上下文管理器

很多资源需要"打开/使用后关闭"：
- 文件读写
- 数据库连接
- 网络连接
- 线程锁
- 临时目录、临时配置

如果手动关闭，容易忘记，或者中间报错导致资源没有释放。上下文管理器能把这些收尾工作交给 Python 自动完成。

### 2. with 语句的基本用法

In [1]:
with open("test.txt", "w", encoding="utf-8") as f:
    f.write("hello")
# 离开 with 块后，文件会自动关闭

### 3. 底层协议：`__enter__` 和 `__exit__`

一个对象能被 `with` 使用，必须实现两个方法：

In [3]:
class MyContext:
    def __enter__(self):
        print("进入上下文")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        """
        :param exc_type: 异常类型
        :param exc_val: 异常实例对象
        :param exc_tb: 异常回溯信息 traceback
        :return: 是否吞异常， True 表示吞异常， False 表示不吞异常，向外抛出
        """
        print("退出上下文")
        return False  # 不吞异常

In [4]:
with MyContext() as c:
    print(c)

进入上下文
退出上下文


执行顺序是：
- 调用 `__enter__`
- 把返回值赋给 `as` 后面的变量
- 执行 `with` 内部代码
- 不管是否异常，都会调用 `__exit__`

### 4. 使用 contextlib 简化写法

In [5]:
from contextlib import contextmanager

@contextmanager
def my_resource():
    print("获取资源")
    try:
        yield "资源对象"
    finally:
        print("释放资源")

with my_resource() as res:
    print("使用资源：", res)

获取资源
使用资源： 资源对象
释放资源


规则很简单：
- yield 之前：相当于 `__enter__`
- yield 本身：把资源交给 `with` 块
- yield 之后：相当于 `__exit__`，通常放在 `finally` 里保证执行

### 5. 易错点
- `as` 后面的变量不是上下文管理器本身，而是 `__enter__` 的返回值
- `__exit__` 默认不要返回 `True`，否则会吞掉异常
- `@contextmanager` 的清理代码最好放在 `try/finally` 中
- `with` 不是“创建上下文管理器”，而是“使用上下文管理器”